# 🧠 CNN-LSTM BISINDO — Preprocessing Pipeline

Notebook ini menjalankan **Step 1 dari pipeline preprocessing**: extract frame mentah dari video.

```
Raw Video
  ↓
[Step 1] Frame Extraction (notebook ini)
         → Blur Filter → Resize → Uniform Sampling
         → Output: frame_manifest.json + frames.npy per video
```

> ℹ️ Augmentasi (flip/rotate/zoom) dan split train/val/test **tidak** dilakukan di sini.
> Keduanya dipindah ke `FeatureExtraction/FeatureExtraction.ipynb` dan dilakukan pada level
> **keypoint** (bukan pixel), karena model final memakai keypoint MediaPipe (T, 126) — augmentasi
> fotometrik pada pixel (brightness/contrast) tidak berpengaruh sama sekali pada keypoint yang
> sudah dinormalisasi, jadi lebih efisien diaugmentasi langsung di ruang keypoint setelah ekstraksi.

**Untuk menambah dataset baru**: edit `Config/dataset_config.py` saja, tidak perlu ubah script lain.


---
## 📦 0. Install Dependencies

In [ ]:
# Install dependencies yang diperlukan
!pip install opencv-python-headless numpy tqdm matplotlib

---
## ⚙️ 1. Setup & Konfigurasi

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── Auto-resolve ROOT repo & tambahkan ke path ────────────────
# PreProcess.ipynb ada di: <ROOT>/PreProcessing/
# Jadi ROOT = 1 level ke atas dari notebook ini
NOTEBOOK_DIR = Path(".").resolve()
ROOT = NOTEBOOK_DIR.parent          # PreProcessing/ → ROOT
PREPROCESS_DIR = NOTEBOOK_DIR       # = ROOT/PreProcessing/

# Tambahkan PreProcessing/ ke sys.path agar bisa import Config & Scripts
if str(PREPROCESS_DIR) not in sys.path:
    sys.path.insert(0, str(PREPROCESS_DIR))

print(f"ROOT repo      : {ROOT}")
print(f"PreProcessing  : {PREPROCESS_DIR}")

from Config.dataset_config import DATASET_CONFIGS, PREPROCESSING_CONFIG
from Scripts.frame_extractor import process_dataset

print("Import berhasil")
print(f"Dataset tersedia: {list(DATASET_CONFIGS.keys())}")


In [ ]:
# ──────────────────────────────────────────────────────────────
# PILIH DATASET DI SINI
# Ganti ke nama dataset lain jika mau proses dataset berbeda
# ──────────────────────────────────────────────────────────────
ACTIVE_DATASET = "WLBisindo"

dataset_cfg = DATASET_CONFIGS[ACTIVE_DATASET]
preprocess_cfg = PREPROCESSING_CONFIG

# Path
RAW_VIDEO_DIR  = dataset_cfg["raw_video_dir"]   # Sesuaikan dengan lokasi video kamu
OUTPUT_DIR     = dataset_cfg["output_dir"]

# Path output
FRAME_MANIFEST = os.path.join(OUTPUT_DIR, "frame_manifest.json")

print(f"
📂 Dataset     : {ACTIVE_DATASET}")
print(f"📂 Raw video   : {RAW_VIDEO_DIR}")
print(f"📂 Output dir  : {OUTPUT_DIR}")
print(f"
⚙️  Preprocessing config:")
print(f"   Target FPS  : {preprocess_cfg['frame']['target_fps']}")
print(f"   Resize      : {preprocess_cfg['frame']['resize']}")
print(f"   Max frames  : {preprocess_cfg['frame']['max_frames']}")


---
## 🎬 2. Step 1 — Frame Extraction

**Input**: Raw `.mp4` videos  
**Proses**: Blur → Resize → Uniform Sampling → Cutting  
**Output**: `frames.npy` per video + `frame_manifest.json`

In [ ]:
# Cek apakah raw video directory ada
if not os.path.exists(RAW_VIDEO_DIR):
    print(f"⚠️  Direktori tidak ditemukan: {RAW_VIDEO_DIR}")
    print(f"\nSesuaikan path 'raw_video_dir' di config/dataset_config.py")
    print(f"Contoh: 'D:/Softwares/GithubProject/CNN-LSTM_BISINDO_Android/WLBisindo'")
else:
    from pathlib import Path
    video_count = len(list(Path(RAW_VIDEO_DIR).rglob("*.mp4")))
    print(f"✅ Direktori ditemukan!")
    print(f"   Total video .mp4: {video_count}")

In [ ]:
# ── Jalankan Frame Extraction ──────────────────────────────────
# overwrite=False → skip video yang sudah diproses (resume-able)
# overwrite=True  → proses ulang semuanya dari awal

frame_stats = process_dataset(
    raw_video_dir=RAW_VIDEO_DIR,
    output_dir=OUTPUT_DIR,
    dataset_config=dataset_cfg,
    preprocessing_config=preprocess_cfg,
    overwrite=False,
)

print(f"\n✅ Frame extraction selesai!")
print(f"   Diproses : {frame_stats['processed']}")
print(f"   Di-skip  : {frame_stats['skipped_short'] + frame_stats['skipped_error']}")

In [ ]:
# Visualisasi distribusi jumlah frame per video
if frame_stats.get("frame_counts"):
    counts = frame_stats["frame_counts"]
    plt.figure(figsize=(10, 4))
    plt.hist(counts, bins=30, color='steelblue', edgecolor='white')
    plt.axvline(np.mean(counts), color='red', linestyle='--', label=f'Mean: {np.mean(counts):.1f}')
    plt.xlabel('Jumlah Frame per Video')
    plt.ylabel('Frekuensi')
    plt.title('Distribusi Frame Count setelah Extraction')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"
📊 Statistik frame:")
    print(f"   Min  : {min(counts)}")
    print(f"   Max  : {max(counts)}")
    print(f"   Mean : {np.mean(counts):.1f}")
    print(f"   Std  : {np.std(counts):.1f}")
    print(f"   (Catatan: target panjang sequence berbeda per model downstream —")
    print(f"    30 frame untuk CNNsa-BiLSTM/keypoint, 16 frame untuk MViTv2/video)")


In [ ]:
# Preview: tampilkan beberapa frame dari video pertama
with open(FRAME_MANIFEST) as f:
    manifest = json.load(f)

if manifest:
    sample = manifest[0]
    frames = np.load(sample["frames_npy"])
    
    print(f"Sample: {sample['video_name']}")
    print(f"Label : {sample['label_name']} (idx={sample['label_idx']})")
    print(f"Frames: {frames.shape}  →  (N_frames, H, W, C)")
    
    # Tampilkan 8 frame
    n_show = min(8, len(frames))
    fig, axes = plt.subplots(1, n_show, figsize=(2.5*n_show, 3))
    for i, ax in enumerate(axes):
        idx = int(i * len(frames) / n_show)
        ax.imshow(frames[idx])
        ax.set_title(f'Frame {idx}', fontsize=8)
        ax.axis('off')
    plt.suptitle(f"Preview: {sample['video_name']}", fontsize=10)
    plt.tight_layout()
    plt.show()

---
## ✅ Ringkasan Preprocessing (Step 1)

Frame extraction selesai. File yang dihasilkan:

```
Datasets/WLBisindo_Processed/
├── frames/                    ← frame per video (.npy), bertingkat per label
└── frame_manifest.json        ← metadata semua video (signer, label, sample, path frame)
```

**Next step**: buka `FeatureExtraction/FeatureExtraction.ipynb` untuk:
1. Ekstraksi keypoint MediaPipe dari `frames.npy` di atas
2. Split train/val/test (stratified per label)
3. Augmentasi keypoint (flip/rotate/zoom) — hanya diterapkan ke train split


In [ ]:
# ── Summary final ──────────────────────────────────────────────
print("="*55)
print(f"  STEP 1 (FRAME EXTRACTION) SELESAI — {ACTIVE_DATASET}")
print("="*55)
print(f"  Diproses      : {frame_stats['processed']}")
print(f"  Skip pendek   : {frame_stats['skipped_short']}")
print(f"  Skip error    : {frame_stats['skipped_error']}")
print(f"  Manifest      : {FRAME_MANIFEST}")
print("="*55)
print("
Lanjut ke FeatureExtraction/FeatureExtraction.ipynb untuk keypoint extraction,")
print("split train/val/test, dan augmentasi.")
